In [ ]:
# Colab setup -- installs SoftMobility when running on Google Colab.
# Tip: For large simulations, switch to a GPU runtime first:
#   Runtime → Change runtime type → T4 GPU (free)
# Then re-run this cell to install the CUDA-enabled JAX.
try:
    import google.colab  # noqa: F401
    import shutil

    if shutil.which("nvidia-smi"):
        %pip install -q -U softmobility "jax[cuda12]"
    else:
        %pip install -q softmobility
except ImportError:
    pass

# Example 01. Sinking rigid body in Stokes regime

A chiral rigid sphere assembly sinks under gravity in Stokes flow.
We choose masses so that the **centre of mass** CoM coincides with the **hydrodynamic mobility centre** $\mathbf{x}_{cm}$; the rotation–translation coupling block of the mobility tensor at that point is then symmetric, and the orientation dynamics admits a closed-form **Jacobi-elliptic** solution.
This tutorial
1. builds that body,
2. Adjust the mass and reference point,
3. runs the library simulation,
4. validates it against the analytical solution.

## Background
The $6 \times 6$ mobility matrix have the block structure

$$\mathbf{M} = \begin{pmatrix} \mathbf{a} & \mathbf{b}^\top \\ \mathbf{b} & \mathbf{c} \end{pmatrix}, $$

where the $3\times 3$ off-diagonal block $\mathbf{b}$ couple translation to rotation.
When $CoM=\mathbf{x}_{cm}$, the body experiences zero gravitational torque about $\mathbf{x}_{cm}$ and $\mathbf{b}$ is symmetric (Gonzalez et al. 2004).

We follow the convention of denoting by $\mathbf{G}$ the *body-frame gravity unit vector* (i.e., the gravity vector expressed in body coordinates),

$$\mathbf{G}(t)=-\mathcal{R}(\boldsymbol{\theta}_0)^{\!\top}\cdot\boldsymbol{e}_3,$$

with $\mathcal{R}(\boldsymbol{\theta}_0)$ the rotation matrix that maps body axes to lab axes.
The capital letter in $\mathbf{G}$ emphasizes that the gravity is expressed in the body frame.

The dynamics of $\mathbf{G}$ reduces to

$$\dot{\mathbf{G}}\;=\;\mathbf{G}\times (\mathbf{b}\cdot\mathbf{G}).$$

This is an Euler-equation type system on the unit sphere; for symmetric $\mathbf{b}$ it integrates exactly in terms of Jacobi elliptic functions (Marsden & Radiu, 2013).

## 1. Defining the chiral four-sphere body

We use a simple chiral 4-sphere assembly: one sphere of radius $0.5$ at the origin (call it sphere 0), and three smaller spheres of radii $(0.1,\,0.3,\,0.5)$ aligned with the body axes $\mathbf{E}_1,\mathbf{E}_2, \mathbf{E}_3$ at the offsets that make each off-axis sphere just touch sphere 0.
The masses are design parameters (we will compute them so CoM = $\mathbf{x}_{cm}$), and gravity enters as a uniform `Field` input.

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
from scipy.spatial.transform import Rotation
from scipy.special import ellipj, ellipk, ellipkinc

from softmobility import FlowBodyRollout, SoftBody, gravity_field, no_flow
from softmobility.classes import figstyle
from softmobility.classes.solver import rotation_matrix

jax.config.update("jax_enable_x64", True)
np.set_printoptions(precision=4, suppress=True, linewidth=120)

# Apply the paper-figure matplotlib aesthetic (white background, black
# box, no grid, Helvetica labels at 11 pt, fixed colour palette,
# orthographic 3D camera). Every figure below inherits this look.
figstyle.apply()

# Every PDF is written under this folder by figstyle.save(...).
FIGDIR = "figures"

In [ ]:
RADII = np.array([0.5, 0.1, 0.3, 0.5])
OFFSETS = np.array([RADII[0] + r for r in RADII[1:]])  # [0.6, 0.8, 1.0]
POSITIONS_BODY = np.array(
    [  # in body frame
        [0.0, 0.0, 0.0],
        [OFFSETS[0], 0.0, 0.0],
        [0.0, OFFSETS[1], 0.0],
        [0.0, 0.0, OFFSETS[2]],
    ]
)

yaml_body = f"""
input_names:  [gravity]
design_names: [mass, xref]

defaults:
  mass0: 1.0
  mass1: 1.0
  mass2: 1.0
  mass3: 1.0
  xref0: 0.0
  xref1: 0.0
  xref2: 0.0

spheres:
  - radius: {RADII[0]}
    position: [xref0, xref1, xref2]
    force:    [mass0*gravity0, mass0*gravity1, mass0*gravity2]
  - radius: {RADII[1]}
    position: [xref0 + {OFFSETS[0]}, xref1, xref2]
    force:    [mass1*gravity0, mass1*gravity1, mass1*gravity2]
  - radius: {RADII[2]}
    position: [xref0, xref1 + {OFFSETS[1]}, xref2]
    force:    [mass2*gravity0, mass2*gravity1, mass2*gravity2]
  - radius: {RADII[3]}
    position: [xref0, xref1, xref2 + {OFFSETS[2]}]
    force:    [mass3*gravity0, mass3*gravity1, mass3*gravity2]
"""

body = SoftBody(yaml_body, verbose=False)

## 2. Hydrodynamic centering

The 6×6 rigid-body mobility decomposes as $\displaystyle \mathbf{M}=\begin{pmatrix}\mathbf{a} & \mathbf{b}^{\!\top} \\ \mathbf{b} & \mathbf{c}\end{pmatrix}.$ The centre of mobility (Kim & Karrila, §5.3.2) is the reference point that *symmetrises* the off-diagonal block $\mathbf{b}$:

$$\mathbf{x}_{cm}\;=\;\tfrac12 \bigl(\mathrm{tr}(\mathbf{c})\,\mathbf{I}-\mathbf{c}\bigr)^{-1}
\,\boldsymbol\varepsilon:(\mathbf{b}-\mathbf{b}^{\!\top}).$$

Once $\mathbf{x}_{cm}$ is known, we redistribute the four masses so that the weighted centre of mass coincides with it.

Finally we shift the reference point by setting `xref = -x_cm` (the body moves so that $\mathbf{x}_{cm}$ ends up at the lab origin), and recompute the mobility there.
At that point $\mathbf{b}$ is symmetric.

In [ ]:
LEVI = np.array(
    [
        [[0.0, 0.0, 0.0], [0.0, 0.0, 1.0], [0.0, -1.0, 0.0]],
        [[0.0, 0.0, -1.0], [0.0, 0.0, 0.0], [1.0, 0.0, 0.0]],
        [[0.0, 1.0, 0.0], [-1.0, 0.0, 0.0], [0.0, 0.0, 0.0]],
    ]
)

# Initial 6x6 mobility (body uncentred, default unit masses).
M0 = np.array(body.compute_rigid_tensors().Mred)
b_blk, c_blk = M0[3:, :3], M0[3:, 3:]
inv_c = np.linalg.inv(np.trace(c_blk) * np.eye(3) - c_blk)
x_cm = 0.5 * np.einsum("ij,jkl,kl->i", inv_c, LEVI, b_blk - b_blk.T)
print("x_cm =", x_cm)

In [ ]:
# Step 1: fix m_0 = 1, solve for m_{1,2,3} so that CoM = x_cm.
A = np.diag(OFFSETS) - np.outer(x_cm, np.ones(3))
m_123 = np.linalg.solve(A, x_cm)
masses = np.concatenate([[1.0], m_123])

# Step 2: renormalise so sum(m) = 1.
masses = masses / masses.sum()

assert np.isclose(masses.sum(), 1.0), masses
assert np.allclose(masses @ POSITIONS_BODY, x_cm, atol=1e-12)
print("masses (sum=1) =", masses)

In [ ]:
# Apply masses + shift xref = -x_cm so that the lab origin coincides
# with both the CoM and the centre of mobility.
body.set_design_defaults(
    new_dict={
        "mass0": float(masses[0]),
        "mass1": float(masses[1]),
        "mass2": float(masses[2]),
        "mass3": float(masses[3]),
        "xref0": float(-x_cm[0]),
        "xref1": float(-x_cm[1]),
        "xref2": float(-x_cm[2]),
    },
    verbose=True,
)

# Recompute and check that the rotation-translation block is symmetric.
M = np.array(body.compute_rigid_tensors().Mred)
b = M[3:, :3]
print("‖b − b.T‖ =", np.linalg.norm(b - b.T))
print("eigvals(b) =", np.linalg.eigvalsh(b))

# Sphere lab positions in the centred configuration.
POSITIONS_CENTRED = (-x_cm)[None, :] + POSITIONS_BODY

### Plotting body geometry

A first sanity figure: the four spheres in their body's frame positions, with the origin which now coincides with both the CoM and the centre of mobility.
The figure is exported as `figures/fig_geometry.pdf` for paper inclusion.

In [ ]:
# Half-width figure with the body axes drawn from the origin (= CoM = x_cm), and
# four red translucent spheres with solid contours.
fig_geom, ax_geom = figstyle.figure_3d(size="half", aspect=1.0)

axis_length = max(RADII[0], RADII[3]) + max(OFFSETS) + 0.15
figstyle.add_body_axes(ax_geom, length=axis_length, show_labels=True)
figstyle.displace_label(ax_geom, "axis_label_E1", text=r"$\mathbf{E}_1$", offset=(0.3, 0, 0))
figstyle.displace_label(ax_geom, "axis_label_E2", text=r"$\mathbf{E}_2$", offset=(0, 0.3, 0))
figstyle.displace_label(ax_geom, "axis_label_E3", text=r"$\mathbf{E}_3$", offset=(0.2, 0, 0))

for centre, radius in zip(POSITIONS_CENTRED, RADII, strict=True):
    figstyle.add_sphere(ax_geom, centre, radius)

bbox = axis_length / 1.1
ax_geom.set_xlim(-bbox, bbox)
ax_geom.set_ylim(-bbox, bbox)
ax_geom.set_zlim(-bbox, bbox)

figstyle.save(fig_geom, "fig_geometry", figdir=FIGDIR)

## 3. Time integration setup and the analytical Jacobi solution

We pick the non-dimensional system $g = m_{\rm tot} = \mu = L = 1$.
The natural timescale is $\tau = 1/\lVert M_2\rVert_2$.

The closed-form Jacobi-elliptic solution to
$$\dot{\mathbf{G}}\;=\;\mathbf{G}\times (\mathbf{b}\cdot\mathbf{G}),$$
on $S^2$ is implemented below.
The parametrization has two cases depending on whether the orbit encircles the eigenvector of the smallest or largest eigenvalue of $\mathbf{b}$, and a sign($\sigma$) branch for the unconstrained component.
The phase $\tau_0$ is recovered via the inverse Jacobi `sn` (using `scipy.special.ellipkinc`).

We pick $\mathbf{G}_0 = (1,1,1)/\sqrt3$ as a clean, off-separatrix initial condition.
(For our specific body, $\mathbf{b}$ has eigenvalues $(-\lambda,\,0,\,+\lambda)$ symmetric about zero, so axis-aligned ICs sit *exactly* on the heteroclinic separatrix where $k\to 1$ and the period diverges; a non-axis-aligned IC stays clear of that pathology.)

In [ ]:
def _inverse_jacobi(sn0, cn0, k2):
    sn0 = float(np.clip(sn0, -1.0, 1.0))
    phi = np.arcsin(sn0)
    u = ellipkinc(phi, k2)
    if cn0 < 0:
        u = 2 * ellipk(k2) - u
    return float(u)


def jacobi_solution(b, G0, time, sep_eps=1e-3):
    """Closed-form G(t) for d G / dt = - G × (b · G) on the unit sphere.

    Returns ``(G_traj, period)`` or ``None`` if the IC lies within
    ``sep_eps`` of the separatrix."""
    eigvals, V = np.linalg.eigh(b)  # ascending: l3 < l2 < l1
    l3, l2, l1 = eigvals
    z0 = V.T @ G0
    a = z0[0] ** 2 * l3 + z0[1] ** 2 * l2 + z0[2] ** 2 * l1
    alpha2 = (a - l3) / (l2 - l3)
    beta2 = (a - l1) / (l2 - l1)
    if abs(alpha2 - beta2) < sep_eps:
        return None
    alpha = np.sqrt(max(alpha2, 0.0))
    beta = np.sqrt(max(beta2, 0.0))
    gamma = np.sqrt((l2 - l3) / (l1 - l3)) * alpha
    delta = np.sqrt((l1 - l2) / (l1 - l3)) * beta

    if alpha < beta:
        # Orbit encircles the eigenvector of the smallest eigenvalue (l3).
        k = alpha / beta
        mu = np.sqrt((l1 - l2) * (l2 - l3)) * beta
        sigma = 1.0 if z0[0] >= 0 else -1.0
        sn0 = sigma * z0[1] / alpha if alpha > 0 else 0.0
        cn0 = -z0[2] / gamma if gamma > 0 else 1.0
        tau0 = _inverse_jacobi(sn0, cn0, k**2)
        tau = tau0 - mu * time
        sn, cn, dn, _ = ellipj(tau, k**2)
        z = np.column_stack([sigma * delta * dn, sigma * alpha * sn, -gamma * cn])
    else:
        # Orbit encircles the eigenvector of the largest eigenvalue (l1).
        k = beta / alpha
        mu = np.sqrt((l1 - l2) * (l2 - l3)) * alpha
        sigma = 1.0 if z0[2] >= 0 else -1.0
        sn0 = -sigma * z0[1] / beta if beta > 0 else 0.0
        cn0 = z0[0] / delta if delta > 0 else 1.0
        tau0 = _inverse_jacobi(sn0, cn0, k**2)
        tau = tau0 - mu * time
        sn, cn, dn, _ = ellipj(tau, k**2)
        z = np.column_stack([delta * cn, -sigma * beta * sn, sigma * gamma * dn])

    period = 4.0 * ellipk(k**2) / mu
    return z @ V.T, float(period)


def G_to_rod(G):
    """Rodrigues vector that brings lab e_3 onto the body-frame vector G."""
    e3 = np.array([0.0, 0.0, 1.0])
    cos_a = float(np.clip(G @ e3, -1.0, 1.0))
    angle = np.arccos(cos_a)
    axis = np.cross(G, e3)
    norm = np.linalg.norm(axis)
    if norm > 1e-10:
        return angle * axis / norm
    return np.zeros(3) if angle < 0.1 else angle * np.array([1.0, 0.0, 0.0])

In [ ]:
# Pick the initial body-frame gravity direction.
G0 = np.array([1.0, 1.0, 1.0]) / np.sqrt(3.0)

# Get the Jacobi period to size the integration window.
_, period = jacobi_solution(b, G0, np.linspace(0.0, 1e4, 200))
print(f"Jacobi period at G0 = {G0}: T = {period:.2f}")

N_PERIODS = 10
N_STEPS = 4000
T_END = N_PERIODS * period
DT = T_END / N_STEPS
time = (np.arange(N_STEPS) + 1) * DT
print(f"T_end = {T_END:.1f}, dt = {DT:.4f}, n_steps = {N_STEPS}")

In [ ]:
# Build the rollout (default scheme = "rk4") and run a single trajectory.
rollout = FlowBodyRollout(body, no_flow(), {"gravity": gravity_field(g=1)})
init_orientation = jnp.asarray(G_to_rod(G0))
positions, oris, _ = rollout.rollout(DT, N_STEPS, init_orientation=init_orientation)
positions = np.array(positions)
oris = np.array(oris)


@jax.jit
def G_from_rod(rod):
    e3 = jnp.array([0.0, 0.0, 1.0])
    return rotation_matrix(rod).T @ e3


@jax.jit
def Gs_from_rods(rods):
    return jax.vmap(G_from_rod)(rods)


G_lib = np.array(Gs_from_rods(jnp.asarray(oris)))
G_ana, _ = jacobi_solution(b, G0, time)
err = float(np.max(np.linalg.norm(G_lib - G_ana, axis=1)))
print(f"max ‖G_rollout − G_analytical‖ = {err:.2e}")

## 4. Orientation in time: numerical vs analytical

The natural quantity for the rollout-vs-analytical comparison is $\mathbf{G}(t) = \mathcal{R}(\boldsymbol{\theta}_0)^{\!\top}\cdot\mathbf{e}_3$ — the analytical Jacobi solution gives this directly.

In [ ]:
@jax.jit
def Rs_from_rods(rods):
    return jax.vmap(rotation_matrix)(rods)


R_traj = np.array(Rs_from_rods(jnp.asarray(oris)))  # (N, 3, 3)
euler_zyx = Rotation.from_matrix(R_traj).as_euler("zyx", degrees=True)

# Restrict to the first ~3 periods so the time series is readable.
keep = time < 3.01 * period
ts = time[keep]
stride = max(1, len(ts) // 50)  # ~50 markers; bump 50 up/down to taste

# Full-width 3-panel figure: G_1, G_2, G_3 with rollout (line) vs
# analytical (sub-sampled markers).
fig_ori, axes_ori = figstyle.subplots(size="full", aspect=658 / 220, ncols=3)
labels = ("$G_1$", "$G_2$", "$G_3$")
for r, comp in enumerate((0, 1, 2)):
    axes_ori[r].plot(
        ts / period,
        G_lib[keep, comp],
        color=figstyle.COLORS["blue"],
        linewidth=1.5,
        label="numerical" if r == 0 else None,
    )
    axes_ori[r].plot(
        ts[::stride] / period,
        G_ana[keep][::stride, comp],
        color=figstyle.COLORS["blue"],
        marker="o",
        linestyle="none",
        markersize=3,
        label="analytical" if r == 0 else None,
    )
    axes_ori[r].set_xlabel("t / T")
    axes_ori[r].set_xticks([0, 1, 2, 3])
    axes_ori[r].set_title(labels[r])
    axes_ori[r].set_ylim(-1.1, 1.1)
    axes_ori[r].set_yticks([])

axes_ori[0].set_yticks([-1, -0.5, 0, 0.5, 1])
axes_ori[0].set_ylabel("$G_i$")
axes_ori[0].legend(loc="lower left", frameon=False)

figstyle.save(fig_ori, "fig_orientation", figdir=FIGDIR)

## 5. Periodic orbits on the unit sphere

To see the global structure we sweep $N_{\mathrm{traj}} = 16$ Fibonacci-distributed initial conditions $\mathbf{G}_0$ on the unit sphere and integrate each.
Each rollout trajectory is overlaid with its analytical Jacobi counterpart (sub-sampled markers), with the body geometry drawn translucently inside the unit sphere as a visual reference.
The diamond markers at $\pm$ eigenvectors of $\boldsymbol{M}_2$ are the steady states of the $\mathbf{G}$ dynamics.

In [ ]:
N_TRAJ = 16
golden = np.pi * (3 - np.sqrt(5))
idx = np.arange(N_TRAJ)
y_fib = 1 - (2 * idx + 1) / N_TRAJ
r_fib = np.sqrt(np.maximum(1 - y_fib**2, 0))
init_Gs = np.column_stack([r_fib * np.cos(golden * idx), r_fib * np.sin(golden * idx), y_fib])
init_rods = np.array([G_to_rod(g) for g in init_Gs])

N_STEPS_orbit = 2000

# Per-IC analytical period. jacobi_solution returns (G_traj, period); we
# only need the period here, so pass a dummy 1-element time grid. Near
# the separatrix it returns None — fall back to the reference `period`
# so the numerical rollout still runs (the analytical overlay below is
# skipped for those ICs).
periods_ana = np.array(
    [period if (sol := jacobi_solution(b, g0, np.array([0.0]))) is None else sol[1] for g0 in init_Gs]
)

# Numerical rollout over 1.2 × T_i for each IC. N_STEPS stays uniform so
# the vmap returns a regular (N_TRAJ, N_STEPS_orbit, 3) array.
DT_per_IC = 1.2 * periods_ana / N_STEPS_orbit


@jax.jit
def run_one(rod0, dt):
    _, oris_one, _ = rollout.rollout(dt, N_STEPS_orbit, init_orientation=rod0)
    return oris_one


oris_all = np.array(
    jax.vmap(run_one)(
        jnp.asarray(init_rods),
        jnp.asarray(DT_per_IC),
    )
)
G_orbit_lib = np.array(jax.vmap(Gs_from_rods)(jnp.asarray(oris_all)))

# Analytical orbit on exactly [0, T_i] with N_STEPS_orbit samples per IC.
G_orbit_ana = []
for g0, T_i in zip(init_Gs, periods_ana, strict=True):
    t_i = (np.arange(N_STEPS_orbit) + 1) * (T_i / N_STEPS_orbit)
    sol = jacobi_solution(b, g0, t_i)
    G_orbit_ana.append(None if sol is None else sol[0])
print(f"analytical available for {sum(s is not None for s in G_orbit_ana)}/{N_TRAJ} ICs")
print(f"per-IC periods: min={periods_ana.min():.3f}, max={periods_ana.max():.3f}, ref={period:.3f}")

In [ ]:
# Full-width 3D figure: orbits of G on S^2 with the body geometry shown
# translucently inside the unit sphere as a visual reference.
fig_orb, ax_orb = figstyle.figure_3d(size="half", aspect=1.0)

# Translucent unit sphere (very faint so trajectories show clearly).
u_s, v_s = np.mgrid[0 : 2 * np.pi : 60j, 0 : np.pi : 30j]
xs_sphere = np.cos(u_s) * np.sin(v_s)
ys_sphere = np.sin(u_s) * np.sin(v_s)
zs_sphere = np.cos(v_s)
ax_orb.plot_surface(
    0.998 * xs_sphere,
    0.998 * ys_sphere,
    0.998 * zs_sphere,
    color=figstyle.COLORS["grey"],
    alpha=0.05,
    linewidth=0,
    antialiased=True,
    shade=False,
)

figstyle.add_body_axes(ax_orb, length=1.15 * 1.1, show_labels=True)
figstyle.displace_label(ax_orb, "axis_label_E1", text=r"$\mathbf{E}_1$", offset=(0.3, 0, 0))
figstyle.displace_label(ax_orb, "axis_label_E2", text=r"$\mathbf{E}_2$", offset=(0, 0.3, 0))
figstyle.displace_label(ax_orb, "axis_label_E3", text=r"$\mathbf{E}_3$", offset=(0.2, 0, 0))


ana_stride = max(1, N_STEPS_orbit // 32)
for i in range(N_TRAJ):
    ax_orb.plot(
        G_orbit_lib[i, :, 0],
        G_orbit_lib[i, :, 1],
        G_orbit_lib[i, :, 2],
        color=figstyle.COLORS["blue"],
        linewidth=1.0,
        label="rollout (RK4)" if i == 0 else None,
    )
    if G_orbit_ana[i] is not None:
        ax_orb.scatter(
            G_orbit_ana[i][::ana_stride, 0],
            G_orbit_ana[i][::ana_stride, 1],
            G_orbit_ana[i][::ana_stride, 2],
            color=figstyle.COLORS["blue"],
            s=5,
            label="analytical Jacobi" if i == 0 else None,
        )

ax_orb.set_xlim(-1.15, 1.15)
ax_orb.set_ylim(-1.15, 1.15)
ax_orb.set_zlim(-1.15, 1.15)

figstyle.save(fig_orb, "fig_s2_orbits", figdir=FIGDIR)

## 7 — Lab-frame trajectory of the centre of mass

Under sustained gravity the body sinks at roughly constant velocity $\langle v_z\rangle$ while rotating, so the lab-frame CoM trajectory is near-vertical with a small horizontal swirl.
To make the rotation– translation coupling visible we plot the trajectory in a **co-moving frame**, $z'(t) = z(t) - \langle v_z\rangle\,t$, with the average vertical drift subtracted.
Grey shadow projections on the three back walls of the bounding cube reveal the orbital pattern of the swirl.

In [ ]:
# Lab-frame CoM trajectory: just plot the raw positions so the body's
# downward sinking under gravity is visible. Grey shadows on the three
# back walls (e_1=min, e_2=min, e_3=min) show the projections.
xs = positions[:, 0]
ys = positions[:, 1]
zs = positions[:, 2]

bounds = (
    [-15, 5],  # range 20
    [-10, 10],  # range 20
    [-5900, 100],  # range 6000 (300 exageration)
)

fig_com, ax = figstyle.figure_3d(size="third", aspect=1.0)
figstyle.add_back_panels(ax, *bounds, color="black", width=1.0)
figstyle.add_shadow(ax, xs, ys, zs, plane="xy_low", bounds=bounds)
figstyle.add_shadow(ax, xs, ys, zs, plane="xz_low", bounds=bounds)
figstyle.add_shadow(ax, xs, ys, zs, plane="yz_low", bounds=bounds)
ax.plot(xs, ys, zs, color=figstyle.COLORS["red"], linewidth=1.5)
ax.set_xlim(*bounds[0])
ax.set_ylim(*bounds[1])
ax.set_zlim(*bounds[2])

figstyle.save(fig_com, "fig_3d_com", figdir=FIGDIR)

## 8 — Summary

We built a chiral 4-sphere body, redistributed its masses so that the centre of mass sits at the hydrodynamic mobility centre $\mathbf{x}_{cm}$, and verified that the rotation–translation coupling block $\mathbf{b}$ becomes symmetric at that reference point.
At this special configuration the body-frame gravity direction $\mathbf{G}$ obeys a closed-form Jacobi-elliptic trajectory on $S^2$, and the library rollout (RK4) reproduces it to $\sim 10^{-9}$ over five Jacobi periods.

The four PDF figures saved alongside the notebook are `figures/fig_geometry.pdf`, `figures/fig_orientation.pdf`, `figures/fig_s2_orbits.pdf`, and `figures/fig_3d_com.pdf`.


## References
O. Gonzalez, A. Graf, and J. Maddocks, Dynamics of a rigid body in a Stokes fluid, J. Fluid Mech. **519**, 133 (2004).

S. Kim and S. J. Karrila, *Microhydrodynamics: principles and selected applications* (Butterworth-Heinemann, 2013).

J. E. Marsden and T. S. Ratiu, *Introduction to mechanics and symmetry: a basic exposition of classical mechanical systems*, Vol. 17 (Springer Science & Business Media, 2013).
